In [9]:
import os
import zipfile
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from google.colab import drive
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, f1_score
import numpy as np
from torch.optim import lr_scheduler

# -------------------------------
# Mount Google Drive
# -------------------------------
drive.mount('/content/drive')

# -------------------------------
# 1️⃣ Unzip the dataset
# -------------------------------
zip_path = "/content/drive/MyDrive/balamced_fer.zip" # Updated path to Google Drive
extract_path = "balanced_fer"

if not os.path.exists(extract_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("Dataset unzipped successfully.")
else:
    print("Dataset already unzipped.")

# -------------------------------
# 2️⃣ Data transforms (safe for small images)
# -------------------------------
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),           # resize all images to 224x224
    transforms.Grayscale(num_output_channels=3),  # convert to 3 channels
    transforms.RandomHorizontalFlip(),       # augmentation
    transforms.RandomRotation(10),           # slight rotation
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# -------------------------------
# 3️⃣ Load datasets using ImageFolder
# -------------------------------
data_dir = extract_path

train_dataset = datasets.ImageFolder(os.path.join(data_dir, "train"), transform=train_transforms)
val_dataset = datasets.ImageFolder(os.path.join(data_dir, "val"), transform=val_test_transforms)
test_dataset = datasets.ImageFolder(os.path.join(data_dir, "test"), transform=val_test_transforms)

# -------------------------------
# 4️⃣ Create DataLoaders
# -------------------------------
batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

# -------------------------------
# 5️⃣ Load pre-trained ResNet18
# -------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Replace final layer for 7 emotion classes
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 7)
model = model.to(device)

# -------------------------------
# 6️⃣ Loss function and optimizer
# -------------------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# -------------------------------
# 6️⃣.1 Learning Rate Scheduler
# -------------------------------
# Define a learning rate scheduler
# This scheduler will decrease the learning rate by a factor of 0.1 every 5 epochs
scheduler = lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

# -------------------------------
# 7️⃣ Training loop (safe, prints progress)
# -------------------------------
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    running_corrects = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(outputs, 1)
        running_corrects += torch.sum(preds == labels.data)

    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = running_corrects.double() / len(train_dataset)

    # Validation
    model.eval()
    val_corrects = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            val_corrects += torch.sum(preds == labels.data)
    val_acc = val_corrects.double() / len(val_dataset)

    print(f"Epoch {epoch+1}/{num_epochs}: "
          f"Train Loss: {epoch_loss:.4f}, Train Acc: {epoch_acc:.4f}, "
          f"Val Acc: {val_acc:.4f}")
    scheduler.step() # Step the scheduler after each epoch

# -------------------------------
# 8️⃣ Test accuracy and saving results
# -------------------------------
model.eval()
test_corrects = 0
all_labels = []
all_preds = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        test_corrects += torch.sum(preds == labels.data)

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

test_acc = test_corrects.double() / len(test_dataset)
print(f"Test Accuracy: {test_acc:.4f}")

# Save the trained model
model_save_path = "/content/drive/MyDrive/new_trained_model.pth"
torch.save(model.state_dict(), model_save_path)
print(f"Trained model saved to {model_save_path}")

# Calculate and save evaluation metrics
metrics_save_path = "/content/drive/MyDrive/new_evaluation_metrics.txt"

with open(metrics_save_path, 'w') as f:
    f.write("\n--- Evaluation Metrics ---\n")
    f.write(f"Test Accuracy: {test_acc:.4f}\n")
    f.write("\nConfusion Matrix:\n")
    f.write(str(confusion_matrix(all_labels, all_preds)))
    f.write("\n\nClassification Report:\n")
    f.write(classification_report(all_labels, all_preds, target_names=test_dataset.classes))
    f.write(f"\nF1-Score (weighted): {f1_score(all_labels, all_preds, average='weighted'):.4f}\n")

print(f"Evaluation metrics saved to {metrics_save_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset already unzipped.
Epoch 1/10: Train Loss: 1.2793, Train Acc: 0.5176, Val Acc: 0.5944
Epoch 2/10: Train Loss: 1.0164, Train Acc: 0.6212, Val Acc: 0.6488
Epoch 3/10: Train Loss: 0.8994, Train Acc: 0.6610, Val Acc: 0.6755
Epoch 4/10: Train Loss: 0.8151, Train Acc: 0.6952, Val Acc: 0.7047
Epoch 5/10: Train Loss: 0.7366, Train Acc: 0.7281, Val Acc: 0.7274
Epoch 6/10: Train Loss: 0.5353, Train Acc: 0.8050, Val Acc: 0.7628
Epoch 7/10: Train Loss: 0.4604, Train Acc: 0.8337, Val Acc: 0.7734
Epoch 8/10: Train Loss: 0.4073, Train Acc: 0.8554, Val Acc: 0.7845
Epoch 9/10: Train Loss: 0.3603, Train Acc: 0.8738, Val Acc: 0.7907
Epoch 10/10: Train Loss: 0.3199, Train Acc: 0.8877, Val Acc: 0.7944
Test Accuracy: 0.7981
Trained model saved to /content/drive/MyDrive/new_trained_model.pth
Evaluation metrics saved to /content/drive/MyDrive/new_evaluation_metrics.txt


### Learning Rate Scheduling
Learning rate scheduling involves adjusting the learning rate during training. This can help the model converge faster and achieve better performance. A common strategy is to reduce the learning rate over time, which allows for larger steps early in training and finer adjustments later on.

Here, we'll use `torch.optim.lr_scheduler.StepLR`, which decays the learning rate by a given factor (`gamma`) every specified number of epochs (`step_size`).

In [7]:
# -------------------------------
# 6️⃣.1 Learning Rate Scheduler
# -------------------------------
from torch.optim import lr_scheduler

# Define a learning rate scheduler
# This scheduler will decrease the learning rate by a factor of 0.1 every 5 epochs
scheduler = lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)